In [1]:
# ============================================
# Colab実装：hit候補（ボール）×pose（人）で hit を絞り込み
# - pose_keypoints.jsonl : 1行=1フレーム
# - events.json          : [{"event":"hit","frame":...,"time_s":...,"x":...,"y":...}, ...]
#
# 出力:
# - filtered_hits.json : 絞り込み後のhitイベント（reason付き）
# ============================================

import json
import math
from typing import Dict, Any, List, Optional, Tuple

from google.colab import drive
drive.mount("/content/drive")

import cv2
import os, json
import datetime

project_folder_name = 'datapingpong-vision-lab/04_hit-detection'
project_path = os.path.join('/content/drive/MyDrive', project_folder_name)

video_path = os.path.join(project_path, "../01_ball-tracking/data/DJI_0056_001.MP4")

out_dir = os.path.join(project_path, "output")
os.makedirs(out_dir, exist_ok=True)
out_jsonl_path = os.path.join(out_dir, "pose_keypoints.jsonl")


# ====== 入力パス（必要に応じて変更）======
POSE_JSONL_PATH = os.path.join(project_path, "../03_pose-estimation/output/pose_keypoints.jsonl")
EVENTS_JSON_PATH = os.path.join(project_path, "../02_bound-detection/annotated_events.json")
OUT_FILTERED_PATH = os.path.join(project_path, "filtered_hits.json")



Mounted at /content/drive


In [ ]:


# ====== パラメータ（探索フェーズ前提：後で調整）======
CFG = {
    # 左右判定の境界（フルフレーム座標系）
    # 例: ROI境界が x=960 なら 960、卓球台中央がわかるならそれでもOK
    "table_center_x": 960.0,

    # 手首-ボール距離しきい値（pixel）
    "hit_hand_dist_px": 60.0,

    # visibility しきい値（0-1）
    "vis_th": 0.5,

    # 肘角度（肩-肘-手首）許容範囲
    "elbow_angle_min": 90.0,
    "elbow_angle_max": 160.0,

    # 手首速度（px/frame）最小
    "wrist_v_min": 8.0,

    # 前後を見るフレーム幅（速度計算に使う）
    "speed_window": 2,  # hit_f - speed_window の手首との差分
}

# ====== MediaPipe Pose landmark id（基本：右利き想定）======
# 右腕: shoulder=12, elbow=14, wrist=16
RIGHT_ARM = {"shoulder": 12, "elbow": 14, "wrist": 16}
# 左腕: shoulder=11, elbow=13, wrist=15
LEFT_ARM  = {"shoulder": 11, "elbow": 13, "wrist": 15}

# --------------------------------------------
# I/O
# --------------------------------------------
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def load_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path: str, obj: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

# --------------------------------------------
# Math helpers
# --------------------------------------------
def norm2(dx: float, dy: float) -> float:
    return math.sqrt(dx*dx + dy*dy)

def angle_abc(a: Tuple[float, float], b: Tuple[float, float], c: Tuple[float, float]) -> float:
    """
    2D内角: ∠ABC (A-B-C)
    """
    bax = a[0] - b[0]
    bay = a[1] - b[1]
    bcx = c[0] - b[0]
    bcy = c[1] - b[1]

    ba = math.sqrt(bax*bax + bay*bay)
    bc = math.sqrt(bcx*bcx + bcy*bcy)
    if ba < 1e-6 or bc < 1e-6:
        return float("nan")

    cosv = (bax*bcx + bay*bcy) / (ba*bc)
    cosv = max(-1.0, min(1.0, cosv))
    return math.degrees(math.acos(cosv))

# --------------------------------------------
# Pose helpers
# --------------------------------------------
def landmarks_to_dict(landmarks: Optional[List[Dict[str, Any]]]) -> Dict[int, Dict[str, Any]]:
    """
    list[{id,x,y,z,vis}] -> dict[id] = {...}
    """
    if landmarks is None:
        return {}
    d = {}
    for lm in landmarks:
        if "id" in lm:
            d[int(lm["id"])] = lm
    return d

def get_lm(lm_dict: Dict[int, Dict[str, Any]], idx: int) -> Optional[Dict[str, Any]]:
    return lm_dict.get(idx)

def visible(lm: Optional[Dict[str, Any]], vis_th: float) -> bool:
    if lm is None:
        return False
    v = lm.get("vis", None)
    if v is None:
        return False
    try:
        return float(v) >= vis_th
    except:
        return False

def pick_player(hit_x: float, table_center_x: float) -> str:
    return "left" if hit_x < table_center_x else "right"

def pick_arm_ids(player: str) -> Dict[str, int]:
    """
    まずは「右利き想定」で右腕を使う。
    将来：利き腕推定を入れるならここを拡張。
    """
    return RIGHT_ARM

def pose_at_frame(pose_by_frame: Dict[int, Dict[str, Any]], frame: int, player: str) -> Optional[Dict[str, Any]]:
    rec = pose_by_frame.get(frame)
    if rec is None:
        return None
    return rec.get(player)

# --------------------------------------------
# Scoring / filtering
# --------------------------------------------
def compute_hit_pose_features(
    hit: Dict[str, Any],
    pose_by_frame: Dict[int, Dict[str, Any]],
    cfg: Dict[str, Any],
) -> Dict[str, Any]:
    """
    hit1件に対して pose由来の特徴量を計算し、判定と理由を返す。
    """
    hit_f = int(hit["frame"])
    hit_x = float(hit["x"])
    hit_y = float(hit["y"])

    player = pick_player(hit_x, cfg["table_center_x"])
    arm = pick_arm_ids(player)

    # --- 現在フレームのpose ---
    p = pose_at_frame(pose_by_frame, hit_f, player)
    if p is None:
        return {
            "player": player,
            "valid": False,
            "reason": {"fail": "pose_missing_frame"}
        }

    lm_dict = landmarks_to_dict(p.get("landmarks"))

    sh = get_lm(lm_dict, arm["shoulder"])
    el = get_lm(lm_dict, arm["elbow"])
    wr = get_lm(lm_dict, arm["wrist"])

    vis_th = cfg["vis_th"]
    if not (visible(el, vis_th) and visible(wr, vis_th)):
        return {
            "player": player,
            "valid": False,
            "reason": {
                "fail": "low_visibility",
                "vis_elbow": None if el is None else el.get("vis"),
                "vis_wrist": None if wr is None else wr.get("vis"),
            }
        }

    # --- 距離 ---
    dist = norm2(hit_x - float(wr["x"]), hit_y - float(wr["y"]))

    # --- 肘角度 ---
    ang = angle_abc(
        (float(sh["x"]), float(sh["y"])) if sh is not None else (float("nan"), float("nan")),
        (float(el["x"]), float(el["y"])),
        (float(wr["x"]), float(wr["y"]))
    )

    # --- 手首速度（前フレームとの差分） ---
    win = int(cfg["speed_window"])
    p_prev = pose_at_frame(pose_by_frame, hit_f - win, player)
    wrist_v = float("nan")
    if p_prev is not None:
        lm_prev = landmarks_to_dict(p_prev.get("landmarks"))
        wr_prev = get_lm(lm_prev, arm["wrist"])
        if visible(wr_prev, vis_th):
            wrist_v = norm2(float(wr["x"]) - float(wr_prev["x"]), float(wr["y"]) - float(wr_prev["y"])) / win

    # --- 判定 ---
    ok_dist = dist <= float(cfg["hit_hand_dist_px"])
    ok_ang  = (not math.isnan(ang)) and (float(cfg["elbow_angle_min"]) <= ang <= float(cfg["elbow_angle_max"]))
    ok_v    = (not math.isnan(wrist_v)) and (wrist_v >= float(cfg["wrist_v_min"]))

    valid = bool(ok_dist and ok_ang and ok_v)

    return {
        "player": player,
        "valid": valid,
        "reason": {
            "dist_px": dist,
            "dist_ok": ok_dist,
            "elbow_angle": ang,
            "angle_ok": ok_ang,
            "wrist_v": wrist_v,
            "wrist_v_ok": ok_v,
            "vis_elbow": el.get("vis"),
            "vis_wrist": wr.get("vis"),
            "arm": arm,
            "speed_window": win,
        }
    }

def filter_hits_with_pose(
    hits: List[Dict[str, Any]],
    pose_by_frame: Dict[int, Dict[str, Any]],
    cfg: Dict[str, Any],
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    returns: (accepted, rejected)
    """
    accepted = []
    rejected = []

    for hit in hits:
        feat = compute_hit_pose_features(hit, pose_by_frame, cfg)

        out = dict(hit)  # shallow copy
        out["player"] = feat["player"]
        out["valid"] = feat["valid"]
        out["pose_reason"] = feat["reason"]

        if feat["valid"]:
            accepted.append(out)
        else:
            rejected.append(out)

    return accepted, rejected

# --------------------------------------------
# Main
# --------------------------------------------
pose_rows = load_jsonl(POSE_JSONL_PATH)
pose_by_frame = {int(r["frame"]): r for r in pose_rows}

events = load_json(EVENTS_JSON_PATH)

# hitだけ抽出（ファイルが hit だけならそのままでOK）
hits = [e for e in events if e.get("event") == "hit"]

accepted, rejected = filter_hits_with_pose(hits, pose_by_frame, CFG)

print("hits total     :", len(hits))
print("accepted (valid):", len(accepted))
print("rejected       :", len(rejected))

# acceptedだけ保存（必要ならrejectedも別ファイルで保存できます）
save_json(OUT_FILTERED_PATH, accepted)
print("saved:", OUT_FILTERED_PATH)

# 任意：rejectedも保存したい場合
# save_json("/content/rejected_hits.json", rejected)


hits total     : 796
accepted (valid): 0
rejected       : 796
saved: /content/drive/MyDrive/datapingpong-vision-lab/04_hit-detection/filtered_hits.json


In [ ]:
import os
import cv2
import json

VIDEO_PATH = os.path.join(project_path, "../01_ball-tracking/data/DJI_0056_001.MP4")
OUTPUT_PATH = os.path.join(project_path, "output_bounce_check.mp4")
EVENT_JSON_PATH = os.path.join(project_path, "annotated_events.json")


with open(OUT_FILTERED_PATH, "r") as f:
    bound_events = json.load(f)
with open(OUT_FILTERED_PATH, "r")as f:
    hit_events_json = json.load(f)

# --- ここから生成（JSON -> bounce_frames / ball_positions） ---

# bounceイベントだけ抽出
bounce_events = [e for e in bound_events if e.get("event") == "bounce" and "frame" in e]

# frame番号リスト（重複除去 + ソート）
bounce_frames = sorted({int(e["frame"]) for e in bounce_events})

hit_events = [e for e in hit_events_json if e.get("event") == "hit" and "frame" in e]
hit_frames = sorted({int(e["frame"]) for e in hit_events})

hit_positions = {}
for e in hit_events:
    fr = int(e["frame"])
    hit_positions[fr] = (float(e["x"]), float(e["y"]))

# frame -> (x, y) を作る
# 優先順位: x,y があればそれを使う。なければ u,v から復元する（0-1正規化）
ball_positions = {}
for e in bounce_events:
    fr = int(e["frame"])
    ball_positions[fr] = (float(e["x"]), float(e["y"]))

# --- ここまで生成 ---
COLOR_BOUNCE = (0, 0, 255)   # 赤
COLOR_HIT    = (0, 255, 0)   # 緑

MARK_DURATION = 60  # フレーム数
RADIUS = 10
# 学習・検証用の解像度
OUT_WIDTH = 640
OUT_HEIGHT = 320

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps == 0:
    fps = 30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (OUT_WIDTH, OUT_HEIGHT)
)
frame_idx = 0

# 「今どのバウンドを表示中か」を管理
active_marks = []  # [(end_frame, (x,y), color)]

bounce_set = set(bounce_frames)

# ログを減らしたい場合はここを True に
VERBOSE = True

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (OUT_WIDTH, OUT_HEIGHT), interpolation=cv2.INTER_AREA)

    # 新しいバウンドが来たら登録
    if frame_idx in bounce_set:
        pos = ball_positions.get(frame_idx)
        if pos is not None:
            x, y = pos
            active_marks.append((frame_idx + MARK_DURATION, (int(x), int(y)), COLOR_BOUNCE))
            if VERBOSE:
                print("append:", active_marks[-1])

    # --- hit ---
    if frame_idx in hit_frames:
        pos = hit_positions.get(frame_idx)
        if pos is not None:
            x, y = pos
            active_marks.append(
                (frame_idx + MARK_DURATION, (int(x), int(y)), COLOR_HIT)
            )

    # 有効期限切れを削除
    active_marks = [
        (end_f, pos, col) for end_f, pos, col in active_marks
        if frame_idx <= end_f
    ]

    # マーキング描画
    for _, (x, y), col in active_marks:
        cv2.circle(frame, (x, y), RADIUS, col, 2)  # thickness=2 → 枠

    # フレーム番号表示
    cv2.putText(
        frame,
        f"Frame: {frame_idx}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (255, 255, 255),
        2
    )

    writer.write(frame)
    frame_idx += 1

    if VERBOSE and frame_idx % 300 == 0:
        print("processed:", frame_idx)

cap.release()
writer.release()

print("saved:", OUTPUT_PATH)


processed: 300
processed: 600
processed: 900
processed: 1200
processed: 1500
processed: 1800
processed: 2100
processed: 2400
processed: 2700
processed: 3000
processed: 3300
processed: 3600
processed: 3900
processed: 4200
processed: 4500
processed: 4800
processed: 5100
processed: 5400
processed: 5700
processed: 6000
processed: 6300
processed: 6600
processed: 6900
processed: 7200
processed: 7500
processed: 7800
processed: 8100
processed: 8400
processed: 8700
processed: 9000
processed: 9300
processed: 9600
processed: 9900
processed: 10200
processed: 10500
processed: 10800
processed: 11100
processed: 11400
processed: 11700
processed: 12000
processed: 12300
processed: 12600
processed: 12900
processed: 13200
processed: 13500
processed: 13800
processed: 14100
processed: 14400
processed: 14700
processed: 15000
processed: 15300
processed: 15600
processed: 15900
processed: 16200
processed: 16500
processed: 16800
processed: 17100
processed: 17400
processed: 17700
processed: 18000
processed: 18300

In [ ]:
# 20260201 ここからまだ

In [ ]:
import json, math, statistics

def load_jsonl(path):
    rows=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

pose_rows = load_jsonl(POSE_JSONL_PATH)
with open(EVENTS_JSON_PATH,"r",encoding="utf-8") as f:
    events = json.load(f)
hits = [e for e in events if e.get("event")=="hit"]

print("pose rows:", len(pose_rows), "hits:", len(hits))
print("pose size:", pose_rows[0].get("size"))

# hitレンジ
hx = [float(h["x"]) for h in hits]
hy = [float(h["y"]) for h in hits]
print("hit x range:", (min(hx), max(hx)))
print("hit y range:", (min(hy), max(hy)))

# pose手首レンジ（左右・右手首16をざっくり）
wx, wy = [], []
for r in pose_rows[:2000]:
    for side in ("left","right"):
        p = r.get(side, {})
        lms = p.get("landmarks")
        if not lms:
            continue
        d = {int(lm["id"]): lm for lm in lms if "id" in lm}
        wr = d.get(16)
        if wr and wr.get("vis", 0) is not None:
            wx.append(float(wr["x"]))
            wy.append(float(wr["y"]))
print("pose wrist x range:", (min(wx), max(wx)) if wx else None)
print("pose wrist y range:", (min(wy), max(wy)) if wy else None)

# 距離分布（左右両方の手首とhitの最小距離を取る：座標系が合ってれば小さくなる）
pose_by_frame = {int(r["frame"]): r for r in pose_rows}

dists = []
for h in hits[:500]:  # まず先頭500件
    f = int(h["frame"])
    rec = pose_by_frame.get(f)
    if not rec:
        continue
    hx, hy = float(h["x"]), float(h["y"])
    best = None
    for side in ("left","right"):
        p = rec.get(side, {})
        lms = p.get("landmarks")
        if not lms:
            continue
        d = {int(lm["id"]): lm for lm in lms if "id" in lm}
        wr = d.get(16)
        if not wr:
            continue
        dist = math.hypot(hx - float(wr["x"]), hy - float(wr["y"]))
        best = dist if best is None else min(best, dist)
    if best is not None:
        dists.append(best)

print("dist samples:", len(dists))
if dists:
    dists_sorted = sorted(dists)
    p50 = statistics.median(dists_sorted)
    p90 = dists_sorted[int(0.9*len(dists_sorted))]
    print("dist min / p50 / p90:", min(dists_sorted), p50, p90)


pose rows: 25906 hits: 796
pose size: {'w': 1920, 'h': 1080}
hit x range: (4.37, 638.3)
hit y range: (67.06, 358.33)
pose wrist x range: (316.1901569366455, 1909.4465446472168)
pose wrist y range: (359.56867933273315, 960.322515964508)
dist samples: 500
dist min / p50 / p90: 124.22197032249399 384.2676878947457 524.1641592988865
